In [7]:
import polars as pl
from procompa import get_project_root
from pathlib import Path

PRJ_ROOT = get_project_root()
data_dir = PRJ_ROOT / "data"

In [8]:
Complex_portal_df = pl.read_csv("/cluster/project/beltrao/kdammer/master_thesis/data/Complex_Portal/Saccharomyces_cerevisiae_ComplexTab.tsv", separator="\t")
#Complex_portal_df.select(pl.col("#Complex ac"), pl.col("Cross references"), pl.col("Subunits (UniProt IDs)")).head(5)

In [9]:
# === Enrich CombFold results CSV with PDB availability + exact-match info ===
import pandas as pd, re

# --- Paths (edit these to match your cluster layout) ---
combfold_csv = data_dir/"Pipeline/third_setup/pdb_present_for_stoi_gr_two_third_setup_pipeline_complexes_combfold_results.csv"
tsv_path     = data_dir/ "Complex_Portal/Saccharomyces_cerevisiae_ComplexTab.tsv"
exact_csv    = data_dir/"complete_complex_pdb_mapping/uniprot_pdb/complex_pdb_exact_match.csv"
out_csv      = data_dir/"complete_complex_pdb_mapping/pdb_present_for_stoi_gr_two_third_setup_pipeline_complexes_combfold_results_enriched.csv"

# --- 1. Load CombFold results ---
cf = pd.read_csv(combfold_csv)
print(f"CombFold CSV: {len(cf)} rows, columns: {list(cf.columns)}")

# Find the complex-ID column (try common names)
cpx_col = None
for candidate in ["#Complex ac", "complex_accession", "Complex ac", "CPX"]:
    if candidate in cf.columns:
        cpx_col = candidate
        break
if cpx_col is None:
    raise SystemExit(f"Could not find complex-ID column. Available: {list(cf.columns)}")
print(f"Using complex-ID column: '{cpx_col}'")

# --- 2. Parse Complex Portal TSV for identity/subset/experimental_evidence tags ---
tsv = pd.read_csv(tsv_path, sep="	")
WWPDB_TAGGED = re.compile(r"^wwpdb:([A-Za-z0-9]{4})\((identity|subset)\)$")
WWPDB_BARE   = re.compile(r"\bwwpdb:([A-Za-z0-9]{4})\b")

cpx_pdb_info = {}  # cpx -> {"tags": set, "pdb_ids": set}
for _, row in tsv.iterrows():
    cpx = str(row["#Complex ac"]).strip()
    if not cpx:
        continue
    tags = set()
    pdbs = set()
    xref = row.get("Cross references", "")
    if isinstance(xref, str):
        for tok in xref.split("|"):
            m = WWPDB_TAGGED.match(tok.strip())
            if m:
                tags.add(m.group(2))
                pdbs.add(m.group(1).upper())
    expev = row.get("Experimental evidence", "")
    if isinstance(expev, str):
        tagged = pdbs.copy()
        for match in WWPDB_BARE.finditer(expev):
            pid = match.group(1).upper()
            if pid not in tagged:
                tags.add("experimental_evidence")
                pdbs.add(pid)
    cpx_pdb_info[cpx] = {"tags": tags, "pdb_ids": pdbs}

# --- 3. Load exact-match CSV ---
exact = pd.read_csv(exact_csv)
exact_info = {}  # cpx -> {"has_exact": bool, "n_exact": int, "pdbs": str}
for _, row in exact.iterrows():
    cpx = row["complex_accession"]
    if row["match_class"] == "exact_match":
        n = row["n_exact_match_pdbs"]
        pdbs = row["all_exact_pdbs"] if pd.notna(row["all_exact_pdbs"]) else ""
        exact_info[cpx] = {"has_exact": True, "n_exact": int(n), "pdbs": pdbs}
    else:
        exact_info[cpx] = {"has_exact": False, "n_exact": 0, "pdbs": ""}

# --- 4. Build enrichment columns ---
has_pdb_col, tags_col, pdb_ids_col = [], [], []
has_exact_col, n_exact_col, exact_pdbs_col = [], [], []

for cpx in cf[cpx_col]:
    cpx = str(cpx).strip()
    info = cpx_pdb_info.get(cpx, {"tags": set(), "pdb_ids": set()})
    has_pdb_col.append("yes" if info["pdb_ids"] else "no")
    tags_col.append(";".join(sorted(info["tags"])) if info["tags"] else "none")
    pdb_ids_col.append(";".join(sorted(info["pdb_ids"])))

    ex = exact_info.get(cpx, {"has_exact": False, "n_exact": 0, "pdbs": ""})
    has_exact_col.append("yes" if ex["has_exact"] else "no")
    n_exact_col.append(ex["n_exact"])
    exact_pdbs_col.append(ex["pdbs"])

cf["has_complex_portal_pdb"]   = has_pdb_col
cf["complex_portal_tags"]      = tags_col
cf["complex_portal_pdb_ids"]   = pdb_ids_col
cf["has_exact_match"]          = has_exact_col
cf["n_exact_match_pdbs"]       = n_exact_col
cf["exact_match_pdb_ids"]      = exact_pdbs_col

# --- 5. Save + summary ---
cf.to_csv(out_csv, index=False)
print(f"Enriched CSV written: {out_csv}")
print(f"=== Summary ===")
print(f"  Total complexes in CombFold CSV: {len(cf)}")
print(f"  Has Complex Portal PDB (any tag): {(cf['has_complex_portal_pdb']=='yes').sum()}")
print(f"    of which identity:        {cf['complex_portal_tags'].str.contains('identity').sum()}")
print(f"    of which subset:          {cf['complex_portal_tags'].str.contains('subset').sum()}")
print(f"    of which experimental_evidence: {cf['complex_portal_tags'].str.contains('experimental_evidence').sum()}")
print(f"  Has SIFTS exact match:      {(cf['has_exact_match']=='yes').sum()}")
print(f"  Has neither:                {((cf['has_complex_portal_pdb']=='no') & (cf['has_exact_match']=='no')).sum()}")

CombFold CSV: 116 rows, columns: ['true_complex', 'predicted_complex', 'size_true', 'size_pred', 'match_count', 'jaccard_similarity', 'split', 'exact_size_match', '#Complex ac', 'confidence_score', 'ComplexConfidence', 'Identifiers (and stoichiometry) of molecules in complex', 'stoichiometry_known', 'solely_proteins', 'comb_fold_submission', 'proteins_with_homodimer_pdb', 'pdb_for_true_stoi', 'combfold_successfully', 'n_assembled_outputs', 'confidence_scores']
Using complex-ID column: '#Complex ac'
Enriched CSV written: /cluster/project/beltrao/kdammer/master_thesis/data/complete_complex_pdb_mapping/pdb_present_for_stoi_gr_two_third_setup_pipeline_complexes_combfold_results_enriched.csv
=== Summary ===
  Total complexes in CombFold CSV: 116
  Has Complex Portal PDB (any tag): 47
    of which identity:        33
    of which subset:          19
    of which experimental_evidence: 4
  Has SIFTS exact match:      46
  Has neither:                61


## does stoichimetry mapping work?

In [10]:
cmp_pdb_mapping = pl.read_csv (data_dir/ "complete_complex_pdb_mapping_v2/complex_all_possible_pdbs.csv")

FileNotFoundError: No such file or directory (os error 2): ...kdammer/master_thesis/data/complete_complex_pdb_mapping_v2/complex_all_possible_pdbs.csv (set POLARS_VERBOSE=1 to see full path)

In [ ]:
Complex_portal_df = Complex_portal_df.select(pl.col("#Complex ac"), pl.col("Identifiers (and stoichiometry) of molecules in complex"))

In [ ]:

# Create the new column
Complex_portal_df = Complex_portal_df.with_columns(
    pl.col("Identifiers (and stoichiometry) of molecules in complex")
    .str.contains(r"\(0\)")
    .not_()
    .alias("stoichiometry_known")
)



In [ ]:
# Perform a left join to bring the column over
complex_all_possible_pdbs = cmp_pdb_mapping.join(
    Complex_portal_df.select(["#Complex ac", "stoichiometry_known", "Identifiers (and stoichiometry) of molecules in complex"]),
    left_on="Complex_ac",
    right_on="#Complex ac",
    how="left"
)

## Filter Complexes that do not already have an exact pdb match

In [ ]:
#mmseq_df = pl.read_csv("/cluster/project/beltrao/kdammer/master_thesis/scripts/mmseq_homology_match/mmseqs/mmseqs_run_e_value_100/mmseqs_results.tsv", separator="\t")
# mmseq_df.head(5)
# # Save the DataFrame to a parquet file
# mmseq_df.write_parquet("/cluster/project/beltrao/kdammer/master_thesis/scripts/mmseq_homology_match/mmseqs/mmseqs_run_e_value_100/mmseqs_results.parquet")

In [ ]:
mmseq_df = pl.read_parquet("/cluster/project/beltrao/kdammer/master_thesis/scripts/mmseq_homology_match/mmseqs/mmseqs_run_e_value_100/results/mmseqs_identity_similarity_e_value_100.parquet")

In [ ]:
CDC28 = mmseq_df.filter(pl.col("protein_id") == "p00546")

In [ ]:
mmseq_df.head(5)

query,target,pident,alnlen,evalue,qlen,tlen,qaln,taln
str,str,f64,i64,f64,i64,i64,str,str
"""p14120""","""6sv4_zY""",100.0,105,3.2940e-61,105,105,"""MAPVKSQESINQKLALVIKSGKYTLGYKST…","""MAPVKSQESINQKLALVIKSGKYTLGYKST…"
"""p14120""","""6xiq_c""",100.0,105,3.2940e-61,105,105,"""MAPVKSQESINQKLALVIKSGKYTLGYKST…","""MAPVKSQESINQKLALVIKSGKYTLGYKST…"
"""p14120""","""6yly_c""",100.0,105,3.2940e-61,105,105,"""MAPVKSQESINQKLALVIKSGKYTLGYKST…","""MAPVKSQESINQKLALVIKSGKYTLGYKST…"
"""p14120""","""6z6k_Lc""",100.0,105,3.2940e-61,105,105,"""MAPVKSQESINQKLALVIKSGKYTLGYKST…","""MAPVKSQESINQKLALVIKSGKYTLGYKST…"
"""p14120""","""7rr5_Lc""",100.0,105,3.2940e-61,105,105,"""MAPVKSQESINQKLALVIKSGKYTLGYKST…","""MAPVKSQESINQKLALVIKSGKYTLGYKST…"


In [ ]:
mmseq_df_p003300 = mmseq_df.filter(pl.col("query").str.contains("p00330"))
mmseq_df_p003300.write_csv("/cluster/project/beltrao/kdammer/master_thesis/scripts/mmseq_homology_match/mmseqs/mmseqs_run_e_value_100/results_p003300.tsv")

In [ ]:
Complex_portal_df = pl.read_csv(data_dir/"Complex_Portal/Saccharomyces_cerevisiae_ComplexTab.tsv", separator="\t")
exact_pdb_mapped_complexes = pl.read_csv(data_dir/"complete_complex_pdb_mapping_v2/excact_pdb_match/complex_all_possible_pdbs.csv")

In [ ]:
exact_pdb_mapped_complexes = exact_pdb_mapped_complexes.filter(~pl.col("identity_match_pdb").is_not_null())

In [ ]:
# get list of compexes that dont have an excat match in pdb, add how many proteins this complex has (and the for how many al lest 50 %)
unmapped_complexes = Complex_portal_df.join(
    exact_pdb_mapped_complexes.select(pl.col("Complex_ac").alias("exact_match_Complex_ac")),
    left_on="#Complex ac",
    right_on="exact_match_Complex_ac",
    how="anti"

how similar are hmmer and mmseq results

In [ ]:
hmmer_p00330 = pl.read_csv("/cluster/project/beltrao/kdammer/master_thesis/scripts/af3-template-analysis_from_ubnAF3Benchmark/af3-template-analysis/chunk_analysis_results/uniqSeqs/p00330/true_identity.csv")
mmseq_df_p003300 = pl.read_csv("/cluster/project/beltrao/kdammer/master_thesis/scripts/mmseq_homology_match/mmseqs/mmseqs_run_e_value_100/results_p003300.tsv")

## Sytematically check simialrity btween mmseq results and jackhmmer

for all 370 protein fow which i ran jackhmmer check how many of the top hits are also in the mmseq 2 results
get all proteins for wwhich i ran jackhmmer, filer mmseq2 results table for those, do left join 

In [5]:
"""
Jackhmmer vs MMseqs2 overlap analysis.

For all proteins that were run through jackhmmer, check how many of the
hits are also found by MMseqs2.

Steps:
  1. Concatenate all true_identity.csv files from per-protein folders
     into one parquet with a "query" column (folder name).
  2. Filter MMseqs2 results for those proteins.
  3. Left join jackhmmer -> MMseqs2 on (query, hit_pdb_id).
  4. Summarize overlap (overall + per-protein).

"""

import argparse
from pathlib import Path

import polars as pl

# ── Defaults (adjust or override via CLI) ──────────────────────────────
DEFAULT_JACKHMMER_DIR = Path(
    "/cluster/project/beltrao/kdammer/master_thesis/scripts/"
    "af3-template-analysis_from_ubnAF3Benchmark/af3-template-analysis/"
    "chunk_analysis_results/uniqSeqs"
)
DEFAULT_MMSEQS_PARQUET = Path(
    "/cluster/project/beltrao/kdammer/master_thesis/scripts/"
    "mmseq_homology_match/mmseqs/mmseqs_run_e_value_100/results/"
    "mmseqs_identity_similarity_e_value_100.parquet"
)

jackhmmer_dir = DEFAULT_JACKHMMER_DIR
mmseqs_parquet = DEFAULT_MMSEQS_PARQUET
output_parquet = jackhmmer_dir / "all_jackhmmer_identity.parquet"
overlap_parquet = jackhmmer_dir / "jackhmmer_mmseqs_overlap.parquet"

# ── Step 1: Concatenate all true_identity_standard_mmseq2_scoring.csv into one parquet ─────
csv_files = sorted(jackhmmer_dir.glob("*/true_identity_standard_mmseq2_scoring.csv"))
print(f"Found {len(csv_files)} true_identity_standard_mmseq2_scoring.csv files")
if not csv_files:
    raise SystemExit(f"No true_identity_standard_mmseq2_scoring.csv files found under {jackhmmer_dir}")

df_jackhmmer = pl.concat([
pl.read_csv(f, schema_overrides={
    "chain_id": pl.Utf8,
    "hit_pdb_id": pl.Utf8,
    "true_identity_percent": pl.Float64,
    "is_high_homology_30": pl.Utf8,
}).with_columns(pl.lit(f.parent.name).alias("query"))
for f in csv_files
])

df_jackhmmer.write_parquet(output_parquet)
print(f"Saved {df_jackhmmer.shape[0]:,} rows to {output_parquet}")
print(f"Unique proteins: {df_jackhmmer['query'].n_unique()}")

# ── Step 2: Load MMseqs2 and filter for jackhmmer proteins ─────────
jackhmmer_proteins = df_jackhmmer["query"].unique().to_list()

mmseqs = pl.read_parquet(mmseqs_parquet)
mmseqs_filtered = mmseqs.filter(pl.col("protein_id").is_in(jackhmmer_proteins))
print(f"\nMMseqs2 total: {mmseqs.height:,} rows")
print(f"MMseqs2 filtered to jackhmmer proteins: {mmseqs_filtered.height:,} rows")

# ── Step 3: Left join jackhmmer -> MMseqs2 ─────────────────────────
mmseqs_for_join = mmseqs_filtered.rename({
"protein_id": "query",
# "identity_percent": "mmseqs_identity_percent",
"similarity_percent": "mmseqs_similarity_percent",
"evalue": "mmseqs_evalue",
"alnlen": "mmseqs_alnlen",
"qlen": "mmseqs_qlen",
"tlen": "mmseqs_tlen",
"is_high_homology": "mmseqs_is_high_homology",
})

joined = df_jackhmmer.join(
mmseqs_for_join,
on=["query", "hit_pdb_id"],
how="left",
)

# ── Step 4: Summary ────────────────────────────────────────────────
n_total = joined.height
n_in_mmseqs = joined.filter(pl.col("mmseqs_similarity_percent").is_not_null()).height
n_not_in_mmseqs = joined.filter(pl.col("mmseqs_similarity_percent").is_null()).height

print(f"\n{'='*60}")
print(f"Overlap summary ({df_jackhmmer['query'].n_unique()} proteins)")
print(f"{'='*60}")
print(f"Total jackhmmer hits:          {n_total:,}")
print(f"Also in MMseqs2:               {n_in_mmseqs:,} ({n_in_mmseqs/n_total*100:.1f}%)")
print(f"NOT in MMseqs2:                {n_not_in_mmseqs:,} ({n_not_in_mmseqs/n_total*100:.1f}%)")

# Hits above 30% in jackhmmer
joined_30 = joined.filter(pl.col("is_high_homology_30") == "YES")
n_30 = joined_30.height
n_30_in = joined_30.filter(pl.col("mmseqs_similarity_percent").is_not_null()).height
n_30_both = joined_30.filter(
(pl.col("mmseqs_similarity_percent").is_not_null())
& (pl.col("mmseqs_similarity_percent") > 30)
).height

print(f"\nJackhmmer hits above 30%:      {n_30:,}")
print(f"  Also in MMseqs2:             {n_30_in:,} ({n_30_in/n_30*100:.1f}%)")
print(f"  Both above 30%:              {n_30_both:,}")

# Per-protein breakdown
per_protein = (
joined.group_by("query")
.agg(
    pl.len().alias("n_jackhmmer_hits"),
    pl.col("mmseqs_similarity_percent").is_not_null().sum().alias("n_in_mmseqs"),
    (pl.col("is_high_homology_30") == "YES").sum().alias("n_jackhmmer_above_30"),
    (
        (pl.col("is_high_homology_30") == "YES")
        & pl.col("mmseqs_similarity_percent").is_not_null()
    ).sum().alias("n_above_30_in_mmseqs"),
)
.sort("query")
)
print(f"\nPer-protein overlap:")
print(per_protein)

# Save
joined.write_parquet(overlap_parquet)
print(f"\nSaved joined result to {overlap_parquet}")
print(f"Shape: {joined.shape}")



Found 440 true_identity_standard_mmseq2_scoring.csv files
Saved 753,542 rows to /cluster/project/beltrao/kdammer/master_thesis/scripts/af3-template-analysis_from_ubnAF3Benchmark/af3-template-analysis/chunk_analysis_results/uniqSeqs/all_jackhmmer_identity.parquet
Unique proteins: 440

MMseqs2 total: 5,305,310 rows
MMseqs2 filtered to jackhmmer proteins: 795,533 rows

Overlap summary (440 proteins)
Total jackhmmer hits:          753,542
Also in MMseqs2:               383,807 (50.9%)
NOT in MMseqs2:                369,735 (49.1%)

Jackhmmer hits above 30%:      109,408
  Also in MMseqs2:             101,161 (92.5%)
  Both above 30%:              100,924

Per-protein overlap:
shape: (440, 5)
┌────────┬──────────────────┬─────────────┬──────────────────────┬──────────────────────┐
│ query  ┆ n_jackhmmer_hits ┆ n_in_mmseqs ┆ n_jackhmmer_above_30 ┆ n_above_30_in_mmseqs │
│ ---    ┆ ---              ┆ ---         ┆ ---                  ┆ ---                  │
│ str    ┆ u32              ┆ u32

In [8]:
high_homology_joined = joined.filter(pl.col("is_high_homology_30") == "YES") 
high_homology_joined

chain_id,hit_pdb_id,true_identity_percent,is_high_homology_30,query,identity_percent,mmseqs_similarity_percent,mmseqs_evalue,mmseqs_alnlen,mmseqs_qlen,mmseqs_tlen,mmseqs_is_high_homology
str,str,f64,str,str,f64,f64,f64,i32,i32,i32,bool
"""A""","""1u4c_A""",100.0,"""YES""","""bub3""",100.0,100.0,1.8680e-230,341,341,349,true
"""A""","""1u4c_B""",100.0,"""YES""","""bub3""",100.0,100.0,1.8680e-230,341,341,349,true
"""A""","""1yfq_A""",100.0,"""YES""","""bub3""",100.0,100.0,1.8680e-230,341,341,342,true
"""A""","""2i3s_A""",100.0,"""YES""","""bub3""",100.0,100.0,1.8680e-230,341,341,349,true
"""A""","""2i3s_C""",100.0,"""YES""","""bub3""",100.0,100.0,1.8680e-230,341,341,349,true
…,…,…,…,…,…,…,…,…,…,…,…
"""A""","""8eti_D""",30.21,"""YES""","""p20447""",30.21,44.93,8.2870e-57,453,523,578,true
"""A""","""8eup_D""",30.21,"""YES""","""p20447""",30.21,44.93,8.2870e-57,453,523,578,true
"""A""","""8euy_D""",30.21,"""YES""","""p20447""",30.21,44.93,8.2870e-57,453,523,578,true
